# 🧩 Text Chunking & Embeddings

This notebook demonstrates:
- Token-aware text chunking using tiktoken
- Creating embeddings with OpenAI/Google
- Indexing in Chroma vector database
- Querying the vector store

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

from chunking import TextChunker
from embeddings import EmbeddingManager
import json
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import os

# Set API key (replace with your key or use environment variable)
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

print("✓ Imports successful")

## 1. Load Processed Papers

Load the papers we processed in the previous notebook.

In [ ]:
# Load processed papers
papers_path = "../data/processed_papers.json"

with open(papers_path, 'r', encoding='utf-8') as f:
    papers = json.load(f)

print(f"✓ Loaded {len(papers)} papers")
for paper in papers:
    print(f"  - {paper['metadata']['title'][:60]}...")

## 2. Initialize Text Chunker

Configure chunking parameters for optimal RAG performance.

In [ ]:
# Initialize chunker with custom parameters
chunker = TextChunker(
    encoding_name="cl100k_base",  # GPT-3.5/4 tokenizer
    chunk_size=400,                # Tokens per chunk
    chunk_overlap=100              # Overlap between chunks
)

print("Text Chunker Configuration:")
print(f"  - Encoding: {chunker.encoding.name}")
print(f"  - Chunk size: {chunker.chunk_size} tokens")
print(f"  - Overlap: {chunker.chunk_overlap} tokens")

## 3. Chunk a Single Section

Let's see how chunking works on a sample text.

In [ ]:
# Get a sample section
sample_text = list(papers[0]['sections'].values())[0]
sample_section = list(papers[0]['sections'].keys())[0]

print(f"Sample section: {sample_section}")
print(f"Original length: {len(sample_text)} characters")
print(f"Token count: {chunker.count_tokens(sample_text)} tokens\n")

# Chunk the text
chunks = chunker.chunk_by_sentences(sample_text)

print(f"✓ Created {len(chunks)} chunks")
print(f"\nFirst chunk preview:")
print(f"  Tokens: {chunks[0]['token_count']}")
print(f"  Text: {chunks[0]['text'][:150]}...")

## 4. Chunk All Papers

Process all papers and create structured chunks with metadata.

In [ ]:
# Chunk all papers
all_chunks = chunker.chunk_multiple_papers(papers)

print(f"\n✓ Total chunks created: {len(all_chunks)}")

# Get statistics
stats = chunker.get_statistics(all_chunks)

print("\n📊 Chunking Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

## 5. Visualize Chunk Distribution

In [ ]:
# Analyze chunk distribution
token_counts = [c['token_count'] for c in all_chunks]
sections = [c['section'] for c in all_chunks]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Token distribution
axes[0].hist(token_counts, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(stats['avg_tokens_per_chunk'], color='red', linestyle='--', 
                linewidth=2, label=f"Mean: {stats['avg_tokens_per_chunk']:.0f}")
axes[0].set_title('Token Distribution Across Chunks', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Chunks per section
section_counts = Counter(sections)
sections_df = pd.DataFrame(list(section_counts.items()), columns=['Section', 'Count'])
sections_df = sections_df.sort_values('Count', ascending=False)

sections_df.plot(x='Section', y='Count', kind='barh', ax=axes[1], color='coral', legend=False)
axes[1].set_title('Chunks per Section', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Chunks')
axes[1].set_ylabel('Section')

plt.tight_layout()
plt.show()

## 6. Save Chunks

Save chunks for later use.

In [ ]:
# Save chunks to JSON
chunks_path = "../data/chunks.json"
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(all_chunks)} chunks to {chunks_path}")

## 7. Create Embeddings

Initialize the embedding manager and create vector representations.

**Note**: This requires an API key. Set it in the first cell or via environment variable.

In [ ]:
# Check if API key is set
if not os.getenv('GOOGLE_API_KEY'):
    print("⚠️ Warning: No GOOGLE_API_KEY found!")
    print("Get free key at: https://makersuite.google.com/app/apikey")
    print("\nSkipping embedding creation for demo purposes.")
    print("To run this section, set your API key in cell 2.")
else:
    # Initialize embedding manager with Google (free & reliable)
    embedding_manager = EmbeddingManager(
        provider="google",
        persist_directory="../chroma_db"
    )
    
    print("✓ Embedding manager initialized (provider: Google Gemini)")

## 8. Create Vector Store

Index all chunks in Chroma vector database.

In [ ]:
if os.getenv('GOOGLE_API_KEY'):
    # Create vector store
    print("Creating vector store... (this may take a minute)")
    
    vectorstore = embedding_manager.create_vectorstore(
        chunks=all_chunks,
        collection_name="sci_papers"
    )
    
    print("\n✅ Vector store created successfully!")
    print(f"   Indexed {len(all_chunks)} chunks")
else:
    print("⚠️ Skipped: No GOOGLE_API_KEY configured")

## 9. Test Similarity Search

Query the vector store to test retrieval.

In [ ]:
if os.getenv('GOOGLE_API_KEY'):
    # Test query
    query = "What are the main findings of the research?"
    
    print(f"Query: '{query}'\n")
    
    # Search
    results = embedding_manager.similarity_search_with_score(query, k=3)
    
    print(f"Top 3 Results:\n")
    for i, (doc, score) in enumerate(results, 1):
        print(f"{'='*60}")
        print(f"Result {i} (Score: {score:.4f})")
        print(f"Paper: {doc.metadata.get('paper_title', 'Unknown')[:50]}...")
        print(f"Section: {doc.metadata.get('section', 'Unknown')}")
        print(f"Text: {doc.page_content[:200]}...")
        print()
else:
    print("⚠️ Skipped: No API key configured")

## 10. Chunk Metadata Analysis

Explore the metadata attached to chunks.

In [ ]:
# Sample chunk with full metadata
sample_chunk = all_chunks[0]

print("📋 Example Chunk Metadata:\n")
for key, value in sample_chunk.items():
    if key != 'text':  # Don't print full text
        print(f"  {key}: {value}")

print(f"\n  text: {sample_chunk['text'][:100]}...")

## Summary

✅ Text chunked into ~400 token pieces with 100 token overlap  
✅ Embeddings created using OpenAI/Google  
✅ Vector store indexed with Chroma  
✅ Similarity search tested successfully  

⏭️ Next: Move to `03_retrieval_rag.ipynb` to build RAG chains